# Lab 03: Linear Systems of Equations and Gaussian Elimination

Engineering problems routinely reduce to solving a **system of linear algebraic equations**:

$$A\mathbf{x} = \mathbf{b}$$

where $A$ is an $n \times n$ matrix of coefficients, $\mathbf{x}$ is the vector of unknowns,
and $\mathbf{b}$ is the known right-hand side. Examples include structural analysis,
circuit simulation, fluid networks, and heat conduction.

## Learning Objectives

By the end of this session, you will be able to:
- Formulate an engineering problem as $A\mathbf{x} = \mathbf{b}$ and interpret each matrix component
- Solve linear systems with `np.linalg.solve` and verify solutions via residuals
- Implement **Naive Gaussian Elimination** (forward elimination + back substitution)
- Identify when naive Gauss fails and apply **partial pivoting** to fix it
- Apply the efficient **Thomas Algorithm** for tridiagonal systems

## Session Outline

| # | Topic | 
|---|-------| 
| 1 | Setting Up and Solving $A\mathbf{x}=\mathbf{b}$ |  
| 2 | Naive Gaussian Elimination |  
| 3 | Partial Pivoting |  
| 4 | Tridiagonal Systems and Thomas Algorithm | 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

---
## 1. Setting Up and Solving $A\mathbf{x} = \mathbf{b}$

### 1.1 Matrix Form

A system of $n$ linear equations in $n$ unknowns:

$$a_{11}x_1 + a_{12}x_2 + \cdots + a_{1n}x_n = b_1$$
$$a_{21}x_1 + a_{22}x_2 + \cdots + a_{2n}x_n = b_2$$
$$\vdots$$
$$a_{n1}x_1 + a_{n2}x_2 + \cdots + a_{nn}x_n = b_n$$

can be written compactly as $A\mathbf{x} = \mathbf{b}$, where:
- $A \in \mathbb{R}^{n \times n}$ — the **coefficient matrix**
- $\mathbf{x} \in \mathbb{R}^n$ — the **unknown vector**
- $\mathbf{b} \in \mathbb{R}^n$ — the **right-hand side vector**

### 1.2 Geometric Interpretation (2×2 case)

Each equation $a_{i1}x_1 + a_{i2}x_2 = b_i$ defines a **line** in the $(x_1, x_2)$ plane.
The solution is the **intersection point**.

| Case | Geometry | Algebra |
|------|----------|---------|
| **Unique solution** | Lines cross at one point | $\det(A) \neq 0$ |
| **No solution** | Parallel lines | Inconsistent system |
| **Infinite solutions** | Coincident lines | Singular matrix, $\det(A) = 0$ |

In [ ]:
# Geometric interpretation of 2x2 linear systems
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x = np.linspace(-1, 5, 300)

# Case 1: Unique solution
# 2x1 + x2 = 5  =>  x2 = 5 - 2*x1
# x1 + 3x2 = 10 =>  x2 = (10 - x1) / 3
# Intersection: x1=1, x2=3
axes[0].plot(x, 5 - 2*x,     'b-', lw=2, label=r'$2x_1 + x_2 = 5$')
axes[0].plot(x, (10 - x)/3,  'r-', lw=2, label=r'$x_1 + 3x_2 = 10$')
axes[0].plot(1, 3, 'ko', ms=10, zorder=5, label='Solution $(1, 3)$')
axes[0].set_title('Unique Solution')
axes[0].set(xlim=[-1, 5], ylim=[-1, 5], xlabel='$x_1$', ylabel='$x_2$')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.4)

# Case 2: No solution (parallel lines)
# x1 + x2 = 3  and  x1 + x2 = 5
axes[1].plot(x, 3 - x, 'b-', lw=2, label=r'$x_1 + x_2 = 3$')
axes[1].plot(x, 5 - x, 'r-', lw=2, label=r'$x_1 + x_2 = 5$')
axes[1].set_title('No Solution (Parallel)')
axes[1].set(xlim=[-1, 5], ylim=[-1, 5], xlabel='$x_1$', ylabel='$x_2$')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.4)

# Case 3: Infinite solutions (same line)
# x1 + x2 = 3  and  2x1 + 2x2 = 6 (same line!)
axes[2].plot(x, 3 - x, 'b-',  lw=5, alpha=0.4, label=r'$x_1 + x_2 = 3$')
axes[2].plot(x, 3 - x, 'r--', lw=2,             label=r'$2x_1 + 2x_2 = 6$')
axes[2].set_title('Infinite Solutions (Coincident)')
axes[2].set(xlim=[-1, 5], ylim=[-1, 5], xlabel='$x_1$', ylabel='$x_2$')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.4)

plt.suptitle('Three Possible Outcomes for a 2×2 Linear System', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### 1.3 Engineering Example: Spring-Mass System

Consider **three masses** connected by **four springs** between two fixed walls:

```
Wall ─[k₁]─ m₁ ─[k₂]─ m₂ ─[k₃]─ m₃ ─[k₄]─ Wall
              ↓F₁         ↓F₂         ↓F₃
```

Applying **force equilibrium** at each mass (positive $x$ to the right):

$$\begin{aligned}
m_1: &\quad (k_1 + k_2)\,x_1 - k_2\,x_2 = F_1 \\
m_2: &\quad -k_2\,x_1 + (k_2 + k_3)\,x_2 - k_3\,x_3 = F_2 \\
m_3: &\quad -k_3\,x_2 + (k_3 + k_4)\,x_3 = F_3
\end{aligned}$$

In matrix form $A\mathbf{x} = \mathbf{b}$:

$$\begin{bmatrix} k_1+k_2 & -k_2 & 0 \\ -k_2 & k_2+k_3 & -k_3 \\ 0 & -k_3 & k_3+k_4 \end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix} =
\begin{bmatrix} F_1 \\ F_2 \\ F_3 \end{bmatrix}$$

> **Note:** The matrix $A$ is **symmetric** and **tridiagonal** — a structure that will become important in Section 4.

In [ ]:
# Spring constants (N/m)
k1, k2, k3, k4 = 40.0, 30.0, 20.0, 10.0

# Applied forces (N)
F1, F2, F3 = 20.0, 40.0, 30.0

# Coefficient matrix (derived from equilibrium equations above)
A = np.array([
    [k1+k2,   -k2,      0],
    [  -k2, k2+k3,    -k3],
    [    0,   -k3,  k3+k4]
])

# --- COMPLETE: assemble the right-hand side vector b = [F1, F2, F3] ---
b = np.array([ ,  ,  ])
# --- COMPLETE ---

print('Coefficient matrix A:')
print(A)
print('\nRight-hand side b:', b)

<!-- SOLUTION
b = np.array([F1, F2, F3])
-->

In [ ]:
# Solve with NumPy
x = np.linalg.solve(A, b)

print('Displacements:')
for i, xi in enumerate(x):
    print(f'  x{i+1} = {xi:.6f} m')

# Verify: compute the residual r = A*x - b (should be ~0)
residual = A @ x - b
print(f'\nResidual A*x - b = {residual}')
print(f'Max |residual|   = {np.max(np.abs(residual)):.2e}  (machine precision: ~2e-16)')

---
## 2. Naive Gaussian Elimination

`np.linalg.solve` is a powerful black box, but understanding the underlying algorithm is essential for:
- Debugging ill-conditioned or singular systems
- Implementing custom or specialized solvers
- Estimating computational cost for large problems

**Gaussian elimination** transforms $A\mathbf{x} = \mathbf{b}$ into an equivalent **upper triangular** system via elementary row operations, then solves by **back substitution**.

### 2.1 Augmented Matrix

Combine $A$ and $\mathbf{b}$ into the **augmented matrix** $[A\,|\,\mathbf{b}]$:

$$\left[\begin{array}{ccc|c}
a_{11} & a_{12} & a_{13} & b_1 \\
a_{21} & a_{22} & a_{23} & b_2 \\
a_{31} & a_{32} & a_{33} & b_3
\end{array}\right]$$

### 2.2 Forward Elimination

For each pivot column $k = 0, 1, \ldots, n-2$:
- For each row $i = k+1, \ldots, n-1$:
  1. Compute the **elimination factor**: $\;f_{ik} = a_{ik}^{(k)}\, /\, a_{kk}^{(k)}$
  2. Update row $i$: $\;\text{row}_i \leftarrow \text{row}_i - f_{ik} \cdot \text{row}_k$

This **zeros out** entry $(i, k)$, producing an upper triangular matrix:

$$\left[\begin{array}{ccc|c}
a_{11}^{(1)} & a_{12}^{(1)} & a_{13}^{(1)} & b_1^{(1)} \\
0 & a_{22}^{(2)} & a_{23}^{(2)} & b_2^{(2)} \\
0 & 0 & a_{33}^{(3)} & b_3^{(3)}
\end{array}\right]$$

### 2.3 Back Substitution

Starting from the **last equation** and working upward:

$$x_{n-1} = \frac{b_{n-1}^{(n)}}{a_{n-1,n-1}^{(n)}}$$

$$x_i = \frac{b_i^{(i+1)} - \displaystyle\sum_{j=i+1}^{n-1} a_{ij}^{(i+1)}\, x_j}{a_{ii}^{(i+1)}}, \quad i = n-2, \ldots, 0$$

### 2.4 Step-by-Step Trace (3×3 Example)

Consider the system:
$$\begin{bmatrix} 2 & 1 & -1 \\ -3 & -1 & 2 \\ -2 & 1 & 2 \end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix} =
\begin{bmatrix} 8 \\ -11 \\ -3 \end{bmatrix}, \qquad \text{true solution: } (x_1, x_2, x_3) = (2, 3, -1)$$

**Augmented matrix:**
$$\left[\begin{array}{ccc|c} 2 & 1 & -1 & 8 \\ -3 & -1 & 2 & -11 \\ -2 & 1 & 2 & -3 \end{array}\right]$$

**Step 1** — pivot column $k=0$, eliminate below row 0:
- $f_{10} = -3/2$: row 1 $\leftarrow$ row 1 $-$ $(-3/2)$ row 0
- $f_{20} = -2/2 = -1$: row 2 $\leftarrow$ row 2 $-$ $(-1)$ row 0

$$\left[\begin{array}{ccc|c} 2 & 1 & -1 & 8 \\ 0 & 1/2 & 1/2 & 1 \\ 0 & 2 & 1 & 5 \end{array}\right]$$

**Step 2** — pivot column $k=1$, eliminate below row 1:
- $f_{21} = 2/(1/2) = 4$: row 2 $\leftarrow$ row 2 $-$ $4$ row 1

$$\left[\begin{array}{ccc|c} 2 & 1 & -1 & 8 \\ 0 & 1/2 & 1/2 & 1 \\ 0 & 0 & -1 & 1 \end{array}\right]$$

**Back substitution:**
- $x_3 = 1/(-1) = -1$
- $x_2 = (1 - (1/2)(-1))/(1/2) = (3/2)/(1/2) = 3$
- $x_1 = (8 - (1)(3) - (-1)(-1))/2 = 4/2 = 2$ ✓

<!--factor =           Aug[i,k] / Aug[k,k]
    Aug[i, k:nb] -=   factor*Aug[k, k:nb] -->

In [ ]:
def gauss_naive(A, b):
    """
    Solve Ax = b using Naive Gaussian Elimination (no pivoting).

    Parameters
    ----------
    A : (n, n) array-like  — coefficient matrix
    b : (n,) array-like    — right-hand side vector

    Returns
    -------
    x   : (n,) ndarray — solution vector
    Aug : (n, n+1) ndarray — augmented matrix after elimination
    """
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    n  = A.shape[0]
    nb = n + 1                         # number of columns in augmented matrix

    # Build augmented matrix [A | b]
    Aug = np.hstack((A, b))

    # ── 1. Forward Elimination ────────────────────────────────────────────────
    for k in range(n - 1):             # pivot column k = 0, 1, ..., n-2
        for i in range(k + 1, n):     # rows to eliminate below pivot
            # --- COMPLETE: compute elimination factor and update row i ---
            factor =                                        # Aug[i,k] / Aug[k,k]
            Aug[i, k:nb] =                                  # row i -= factor * row k 
            # --- COMPLETE ---

    # ── 2. Back Substitution ──────────────────────────────────────────────────
    x = np.zeros(n)
    # --- COMPLETE: solve for x[n-1] first, then work backwards ---
    x[n-1] =                                          # Aug[n-1, nb-1] / Aug[n-1, n-1]
    for i in range(n - 2, -1, -1):
        x[i] =                                        # (Aug[i, nb-1] - Aug[i, i+1:n] @ x[i+1:n]) / Aug[i, i]
    # --- COMPLETE ---

    return x, Aug

<!-- SOLUTION
factor        = Aug[i, k] / Aug[k, k]
Aug[i, k:nb]  = Aug[i, k:nb] - factor * Aug[k, k:nb]
x[n-1]        = Aug[n-1, nb-1] / Aug[n-1, n-1]
x[i]          = (Aug[i, nb-1] - Aug[i, i+1:n] @ x[i+1:n]) / Aug[i, i]
-->

In [ ]:
# Test gauss_naive on the 3x3 example traced above
A_test = np.array([[ 2,  1, -1],
                   [-3, -1,  2],
                   [-2,  1,  2]], dtype=float)
b_test = np.array([8.0, -11.0, -3.0])

x_sol, Aug_result = gauss_naive(A_test, b_test)

print('Upper triangular augmented matrix after elimination:')
print(np.round(Aug_result, 6))
print()
print(f'Solution x  = {x_sol}')
print(f'Expected    = [2, 3, -1]')
print(f'Residual    = {A_test @ x_sol - b_test}')

In [ ]:
# Apply gauss_naive to the spring-mass system
x_gauss, _ = gauss_naive(A, b)
x_numpy     = np.linalg.solve(A, b)

print('Spring-mass displacements:')
print(f'  Gauss Elimination : {x_gauss}')
print(f'  np.linalg.solve   : {x_numpy}')
print(f'  Max difference    : {np.max(np.abs(x_gauss - x_numpy)):.2e}')

### 2.5 Computational Cost

For an $n \times n$ system, Naive Gaussian Elimination requires approximately:

| Phase | Multiplications / Divisions | Additions / Subtractions |
|-------|-----------------------------|--------------------------|
| Forward elimination | $\approx n^3/3$ | $\approx n^3/3$ |
| Back substitution   | $\approx n^2/2$ | $\approx n^2/2$ |
| **Total** | $\approx \mathbf{n^3/3}$ | $\approx \mathbf{n^3/3}$ |

> **Key takeaway:** Gaussian elimination is $O(n^3)$. Doubling $n$ **multiplies** runtime by 8. For $n = 10{,}000$, this is $\approx 3 \times 10^{11}$ operations — which is why special structure (e.g., tridiagonal, sparse) matters enormously.

In [ ]:
# O(n^3) scaling: measure numpy.linalg.solve for different n
sizes = [50, 100, 200, 400, 800]
times_numpy = []

for n in sizes:
    # Diagonally dominant random matrix (guaranteed nonsingular)
    A_rand = np.random.randn(n, n) + n * np.eye(n)
    b_rand = np.random.randn(n)
    t0 = time.perf_counter()
    np.linalg.solve(A_rand, b_rand)
    times_numpy.append(time.perf_counter() - t0)

plt.figure(figsize=(7, 4))
plt.loglog(sizes, times_numpy, 'bs-', ms=7, label='numpy.linalg.solve')
n_ref = np.array(sizes, dtype=float)
plt.loglog(sizes, times_numpy[0] * (n_ref / sizes[0])**3, 'r--', label=r'$O(n^3)$ reference')
plt.xlabel('Matrix size $n$')
plt.ylabel('Wall-clock time (s)')
plt.title('Gaussian Elimination: $O(n^3)$ Scaling')
plt.legend()
plt.grid(True, which='both', ls=':', alpha=0.5)
plt.tight_layout()
plt.show()

---
## 3. Partial Pivoting

### 3.1 The Zero-Pivot Problem

Naive Gaussian Elimination **divides by the pivot element** $a_{kk}^{(k)}$ at each step.
If the pivot is **zero**, the algorithm fails; if it is **very small**, roundoff errors are amplified — exactly like catastrophic cancellation.

**Example:** The system
$$\begin{bmatrix} 0 & -3 & 7 \\ 1 & 2 & -1 \\ 5 & -2 & 0 \end{bmatrix}\mathbf{x} =
\begin{bmatrix} 4 \\ 0 \\ 3 \end{bmatrix}$$
has a **zero leading pivot** $a_{11} = 0$ — naive elimination fails immediately with division by zero.

In [ ]:
A_bad = np.array([[0.0, -3.0,  7.0],
                  [1.0,  2.0, -1.0],
                  [5.0, -2.0,  0.0]])
b_bad = np.array([4.0, 0.0, 3.0])

print('True solution (via numpy):')
print('  x =', np.linalg.solve(A_bad, b_bad))

print('\nNaive Gaussian Elimination result:')
x_fail, Aug_fail = gauss_naive(A_bad, b_bad)
print('  x =', x_fail)
print('Augmented matrix after elimination:')
print(Aug_fail)

### 3.2 Partial Pivoting

**Partial pivoting** reorders rows before each elimination step:

At pivot step $k$:
1. Find the row $r \geq k$ with the **largest absolute value** in column $k$
2. **Swap** rows $k$ and $r$ in the augmented matrix
3. Proceed with forward elimination

This guarantees:
- The pivot is **never zero** (as long as the system has a unique solution)
- The elimination factor $|f_{ik}| = |a_{ik}| / |a_{kk}| \leq 1$, preventing error amplification

The resulting algorithm is called **Gaussian Elimination with Partial Pivoting (GEPP)**.

**Row swap in augmented matrix notation:**
$$\left[\begin{array}{ccc|c} \cdots & \mathbf{0} & \cdots & \cdots \\ \cdots & \mathbf{big} & \cdots & \cdots \end{array}\right]
\xrightarrow{\text{swap}}
\left[\begin{array}{ccc|c} \cdots & \mathbf{big} & \cdots & \cdots \\ \cdots & \mathbf{0} & \cdots & \cdots \end{array}\right]$$

In [ ]:
def gauss_pivot(A, b):
    """
    Solve Ax = b using Gaussian Elimination with Partial Pivoting.

    Parameters
    ----------
    A : (n, n) array-like  — coefficient matrix
    b : (n,) array-like    — right-hand side vector

    Returns
    -------
    x   : (n,) ndarray — solution vector
    Aug : (n, n+1) ndarray — augmented matrix after elimination
    """
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    n  = A.shape[0]
    nb = n + 1
    Aug = np.hstack((A, b))

    for k in range(n - 1):
        # ── Partial pivoting ────────────────────────────────────────────────
        # Find the row index (>= k) with the largest |value| in column k
        imax = np.argmax(np.abs(Aug[k:, k])) + k
        if imax != k:
            # --- COMPLETE: swap row k and row imax in the augmented matrix ---
            Aug[[k, imax]] =                        # swap rows
            # --- COMPLETE ---

        # ── Forward elimination (same as naive) ─────────────────────────────
        for i in range(k + 1, n):
            factor = Aug[i, k] / Aug[k, k]
            Aug[i, k:nb] = Aug[i, k:nb] - factor * Aug[k, k:nb]

    # ── Back substitution (same as naive) ───────────────────────────────────
    x = np.zeros(n)
    x[n-1] = Aug[n-1, nb-1] / Aug[n-1, n-1]
    for i in range(n - 2, -1, -1):
        x[i] = (Aug[i, nb-1] - Aug[i, i+1:n] @ x[i+1:n]) / Aug[i, i]

    return x, Aug

<!-- SOLUTION
Aug[[k, imax]] = Aug[[imax, k]]
-->

In [ ]:
# Test gauss_pivot on the previously-failing zero-pivot example
x_pivot, Aug_pivot = gauss_pivot(A_bad, b_bad)
x_ref              = np.linalg.solve(A_bad, b_bad)

print('Gauss with partial pivoting:')
print(f'  x       = {x_pivot}')
print(f'  x (ref) = {x_ref}')
print(f'  Max error: {np.max(np.abs(x_pivot - x_ref)):.2e}')
print()
print('Upper triangular augmented matrix after pivoting:')
print(np.round(Aug_pivot, 6))

### 3.3 Ill-Conditioning and the Condition Number

Even when a unique solution exists, some systems are **ill-conditioned**: small perturbations in $\mathbf{b}$ (e.g., measurement noise) cause large changes in $\mathbf{x}$.

The **condition number** $\kappa(A) = \|A\|\,\|A^{-1}\|$ quantifies this sensitivity:

| $\kappa(A)$ | Interpretation |
|-------------|----------------|
| $\approx 1$ | Well-conditioned, reliable solution |
| $10^6$ | Solution may lose $\sim$6 digits of accuracy |
| $\approx 1/\varepsilon_{\text{mach}} \approx 4.5 \times 10^{15}$ | Effectively singular |

A rule of thumb: if $\kappa(A) \approx 10^k$, you lose approximately $k$ significant digits in the solution.

In [ ]:
# Condition numbers for well- and ill-conditioned matrices
A_good = np.array([[4.0, 1.0], [2.0, 3.0]])
print(f'Well-conditioned 2x2:  kappa = {np.linalg.cond(A_good):.2f}')

# The Hilbert matrix H[i,j] = 1/(i+j+1) is famously ill-conditioned
def hilbert(n):
    return np.array([[1.0 / (i + j + 1) for j in range(n)] for i in range(n)])

print('\nHilbert matrix condition numbers:')
for n in [3, 5, 7, 10, 12]:
    kappa = np.linalg.cond(hilbert(n))
    print(f'  n={n:2d}:  kappa = {kappa:.2e}')

# Demonstrate sensitivity: perturb b slightly and observe the effect on x
n = 10
H = hilbert(n)
x_true = np.ones(n)
b_exact = H @ x_true

rng = np.random.default_rng(42)
delta_b = rng.standard_normal(n) * 1e-7
x_perturbed = np.linalg.solve(H, b_exact + delta_b)

print(f'\nHilbert n={n}  (kappa = {np.linalg.cond(H):.1e}):')
print(f'  ||delta b|| = {np.linalg.norm(delta_b):.2e}')
print(f'  ||delta x|| = {np.linalg.norm(x_perturbed - x_true):.2e}')
print(f'  Amplification factor = {np.linalg.norm(x_perturbed - x_true) / np.linalg.norm(delta_b):.1e}')

---
## 4. Tridiagonal Systems and the Thomas Algorithm

### 4.1 Tridiagonal Structure

Many engineering problems (1D heat conduction, beam deflection, finite-difference PDEs) produce **tridiagonal** systems — only the main diagonal, one subdiagonal, and one superdiagonal are nonzero:

$$\begin{bmatrix}
f_0 & g_0 &        &        &     \\
e_1 & f_1 & g_1    &        &     \\
    & e_2 & f_2    & g_2    &     \\
    &     & \ddots & \ddots & \ddots \\
    &     &        & e_{n-1} & f_{n-1}
\end{bmatrix}\mathbf{x} = \mathbf{r}$$

Applying Gaussian elimination to this structure produces an algorithm with **$O(n)$ operations** instead of $O(n^3)$.

### 4.2 Thomas Algorithm

**Forward sweep** — eliminate the subdiagonal ($e$ entries), for $k = 1, 2, \ldots, n-1$:

$$\text{factor} = \frac{e_k}{f_{k-1}}$$
$$f_k \leftarrow f_k - \text{factor} \cdot g_{k-1}$$
$$r_k \leftarrow r_k - \text{factor} \cdot r_{k-1}$$

**Back substitution**, for $k = n-2, n-3, \ldots, 0$:

$$x_{n-1} = \frac{r_{n-1}}{f_{n-1}}, \qquad x_k = \frac{r_k - g_k\, x_{k+1}}{f_k}$$

In [ ]:
def thomas(e, f, g, r):
    """
    Solve a tridiagonal system using the Thomas Algorithm.

    Parameters (all 1-D arrays of length n)
    ----------
    e : subdiagonal    e[0] unused, e[1]..e[n-1] active
    f : main diagonal  f[0]..f[n-1]  (modified internally)
    g : superdiagonal  g[0]..g[n-2] active, g[n-1] unused
    r : right-hand side r[0]..r[n-1]  (modified internally)

    Returns
    -------
    x : (n,) ndarray — solution vector
    """
    f = np.array(f, dtype=float)   # work on copies
    r = np.array(r, dtype=float)
    n = len(f)

    # ── Forward sweep: eliminate subdiagonal ─────────────────────────────────
    for k in range(1, n):
        # --- COMPLETE: compute factor and update f[k] and r[k] ---
        factor =                                     # e[k] / f[k-1]
        f[k] =                                       # f[k] - factor * g[k-1]
        r[k] =                                       # r[k] - factor * r[k-1]
        # --- COMPLETE ---

    # ── Back substitution ────────────────────────────────────────────────────
    x = np.zeros(n)
    x[-1] = r[-1] / f[-1]
    for k in range(n - 2, -1, -1):
        x[k] = (r[k] - g[k] * x[k+1]) / f[k]

    return x

<!-- SOLUTION
factor = e[k] / f[k-1]
f[k]   = f[k] - factor * g[k-1]
r[k]   = r[k] - factor * r[k-1]
-->

### 4.3 Application: Steady-State 1D Heat Conduction

A 1D rod of length $L = 1\,\text{m}$ with boundary conditions $T(0) = T_L$ and $T(L) = T_R$.
Dividing into $n+1$ equal segments ($\Delta x = L/(n+1)$) and applying the finite-difference approximation
$d^2T/dx^2 \approx (T_{i-1} - 2T_i + T_{i+1})/\Delta x^2 = 0$ gives:

$$T_{i-1} - 2T_i + T_{i+1} = 0, \quad i = 1, \ldots, n$$

This is a tridiagonal system with $e_i = g_i = 1$, $f_i = -2$. The boundary conditions enter through
$r_1 = -T_L$ and $r_n = -T_R$ (all other $r_i = 0$).

In [ ]:
def heat_conduction_1d(n_interior, T_left=50.0, T_right=150.0):
    """Solve steady-state 1D heat conduction on n_interior interior nodes."""
    n = n_interior
    f =  -2.0 * np.ones(n)   # main diagonal
    e =   1.0 * np.ones(n)   # subdiagonal  (e[0] unused)
    g =   1.0 * np.ones(n)   # superdiagonal (g[n-1] unused)
    r =   np.zeros(n)

    # Boundary conditions modify the first and last RHS entries
    r[0]  -= T_left
    r[-1] -= T_right

    T_interior = thomas(e, f, g, r)

    x_all = np.linspace(0, 1, n + 2)
    T_all = np.concatenate([[T_left], T_interior, [T_right]])
    return x_all, T_all


# Exact solution: linear profile (for zero internal heat generation)
T_left, T_right = 50.0, 150.0
x_exact = np.linspace(0, 1, 500)
T_exact = T_left + (T_right - T_left) * x_exact

plt.figure(figsize=(8, 4))
for n in [3, 6, 15]:
    x, T = heat_conduction_1d(n)
    plt.plot(x, T, 'o-', ms=6, label=f'Thomas, {n} interior nodes')
plt.plot(x_exact, T_exact, 'k--', lw=2, label='Exact (linear)')
plt.xlabel('Position $x$ (m)')
plt.ylabel('Temperature $T$ (°C)')
plt.title('Steady-State 1D Heat Conduction — Thomas Algorithm')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Performance: Thomas (O(n)) vs numpy full solve (O(n^3)) for tridiagonal matrices
sizes_tri = [100, 500, 2000, 5000]
times_thomas = []
times_full   = []

for n in sizes_tri:
    f_diag = -2.0 * np.ones(n)
    e_diag =  1.0 * np.ones(n)
    g_diag =  1.0 * np.ones(n)
    r_diag = np.random.randn(n)

    # Build explicit full tridiagonal matrix for numpy
    A_tri = np.diag(f_diag) + np.diag(e_diag[1:], -1) + np.diag(g_diag[:-1], 1)

    t0 = time.perf_counter()
    thomas(e_diag, f_diag, g_diag, r_diag)
    times_thomas.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    np.linalg.solve(A_tri, r_diag)
    times_full.append(time.perf_counter() - t0)

print(f'{"n":>6}  {"Thomas (s)":>12}  {"Full Gauss (s)":>16}  {"Speedup":>10}')
print('-' * 52)
for i, n in enumerate(sizes_tri):
    speedup = times_full[i] / times_thomas[i]
    print(f'{n:>6}  {times_thomas[i]:>12.6f}  {times_full[i]:>16.6f}  {speedup:>10.1f}x')

---
## Summary

### Methods Comparison

| Method | Cost | When to Use | Notes |
|--------|------|-------------|-------|
| **`np.linalg.solve`** | $O(n^3)$ | Any dense $n \times n$ system | Reliable black box (LAPACK) |
| **Naive Gauss Elimination** | $O(n^3)$ | Teaching / understanding | Fails at zero pivot |
| **Gauss with Partial Pivoting** | $O(n^3)$ | General dense systems | Numerically stable |
| **Thomas Algorithm** | $O(n)$ | Tridiagonal systems | ~$n^2/3$ times faster |

### Key Formulas

| Concept | Formula / Rule |
|---------|----------------|
| **Elimination factor** | $f_{ik} = a_{ik}^{(k)}\, /\, a_{kk}^{(k)}$ |
| **Row update** | $\text{row}_i \leftarrow \text{row}_i - f_{ik} \cdot \text{row}_k$ |
| **Back substitution** | $x_i = \left(b_i - \sum_{j>i} a_{ij}\, x_j\right) / a_{ii}$ |
| **Partial pivoting** | Swap to largest $|$pivot$|$ in column before each step |
| **Residual check** | $\mathbf{r} = A\mathbf{x} - \mathbf{b} \approx \mathbf{0}$ |
| **Condition number** | $\kappa(A) = \|A\|\,\|A^{-1}\|$ — measures solution sensitivity |